# Developing with ChatGPT (new OpenAI Responses API)

This notebook explores different ways to use GPT models as simple assistants.

**Hard Constraints**

- One prompt
- One model call
- No memory
- No retrieval

**Main Scenarios**

- Summarizing text
- Detecting sentiment in a text
- Classifying mental states
- Translation
- Changing tone
- Troubleshooting software problems from an error message

## Setup

In [11]:
import os
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv()) # read local .env file
client = OpenAI()
client.api_key = os.environ["OPENAI_API_KEY"]

In [12]:
# Get completion for a single prompt (str) or a chat (list[dict])
def get_completion(
    prompt,
    model="gpt-4o-mini",
    temperature=0,
    max_output_tokens=100
):
    response = client.responses.create(
        input=prompt,
        model=model,
        max_output_tokens=max_output_tokens,
        temperature=temperature
    )

    return response.output[0].content[0].text

## Text summarization

In [3]:
DEF_GPT_MODEL = "gpt-4o-mini"
DELIMITER = "````"
MAX_WORDS = 100
MAX_CHARS_IN = 20_000

In [4]:
SUMMARY_PROMPT = """\
Your task is to summarize a text into bullet point items. The input text will be delimited by {f_delimiter}. \
Limit your response to {f_max_words} words and don't output incomplete sections and sentences. \
Text to summarize : {f_delimiter}{f_text}{f_delimiter}"""

In [5]:
def summarize_text(
    text: str,
    delimiter=DELIMITER,
    max_words=MAX_WORDS,
    max_chars_in = MAX_CHARS_IN,
    model: str=DEF_GPT_MODEL,
    debug=False) -> str:
    """ Uses given GPT model to summarize input text into bullet list format.
    If input text is longer than expected, it will be truncated to configured
    maximum value. """

    if not text:
        raise ValueError("The value of 'text' parameter cannot be None or an empty string.")

    text = text.replace("\n", " ").strip()
    if len(text) > max_chars_in:
        text = text[:max_chars_in]

    max_tokens = int(max_words * 4. / 3)  # simple approximation
    prompt = SUMMARY_PROMPT.format(
        f_delimiter=delimiter, f_max_words=max_words, f_text=text)
    if debug:
        print(f"Input prompt :\n{prompt}")
    response = get_completion(prompt, model=model, max_output_tokens=max_tokens)

    return response

In [6]:
def summarize_file(
    path,
    delimiter=DELIMITER,
    max_words=MAX_WORDS,
    max_chars_in = MAX_CHARS_IN,
    model: str=DEF_GPT_MODEL) -> str:
    """ Uses given GPT model to summarize contents of given text file
    into bullet list format. If text content is longer than expected,
    the rest of file contents will be truncated. Raises error if given
    file is binary, invalid or inaccessible. """

    try:
        with open(path, "r") as file:
            text = file.read()

        summary = summarize_text(
            text, delimiter=delimiter, max_words=max_words,
            max_chars_in=max_chars_in, model=model)
        return summary
    except Exception as ex:
        print(f"[ERROR] {ex}")

In [7]:
SUMMARY_PROMPT.format(
    f_delimiter=DELIMITER, f_max_words=MAX_WORDS, f_text="Some text")

"Your task is to summarize a text into bullet point items. The input text will be delimited by ````. Limit your response to 100 words and don't output incomplete sections and sentences. Text to summarize : ````Some text````"

In [8]:
sample_text = f"""
What is Agentic AI?

Agentic AI is an autonomous AI system that can act independently to achieve pre-determined goals. Traditional software follows pre-defined rules, and traditional artificial intelligence also requires prompting and step-by-step guidance. However, agentic AI is proactive and can perform complex tasks without constant human oversight. "Agentic" indicates agency — the ability of these systems to act independently, but in a goal-driven manner.

AI agents can communicate with each other and other software systems to automate existing business processes. But beyond static automation, they make independent contextual decisions. They learn from their environment and adapt to changing conditions, enabling them to perform sophisticated workflows with accuracy.

For example, an agentic AI system can optimize employee shift schedules. If an employee is off sick, the agent can communicate with other employees and readjust the schedule while still meeting project resource and time requirements.
What are the characteristics of agentic AI systems?

Here are the key features of an agentic AI system.
Proactive

Agentic AI acts proactively rather than waiting for direct input. Traditional systems are reactive, responding only when triggered and following predefined workflows. In contrast, agentic systems anticipate needs, identify emerging patterns, and take initiative to address potential issues before they escalate. Their proactive behavior is driven by environmental awareness and their ability to evaluate outcomes against long-term goals.

For instance, in a supply chain setting, a traditional logistics platform updates delivery statuses when a user checks in or through periodic notifications. An agentic AI system, however, can monitor inventory levels, track weather conditions, and anticipate shipping delays. It can proactively raise alerts and even reroute shipments to reduce downtime.
Adaptable

A key feature of agentic AI is its ability to adapt to changing environments and specific domains. Traditional SaaS solutions are built to scale across industries and handle repetitive tasks, but they often lack the depth to understand unique domain-specific situations. Agentic systems fill this gap by using context awareness and domain knowledge, enabling AI agents to respond intelligently. They adjust their actions based on real-time input and can handle complex scenarios that standard solutions cannot.

For example, while a generic customer service platform might respond with predefined answers, an agentic AI system supporting a healthcare provider understands medical terminology and complies with healthcare regulations. It can adapt to evolving patient concerns and delivers more accurate, context-sensitive support.
Collaborative

Agentic AI is designed to collaborate with humans and with other agentic AI systems. AI agents work as part of a broader team. They can understand shared goals, interpret human intent, and coordinate actions accordingly. They work well in settings that require human oversight or decision-making by considering inputs from multiple sources.

For example, a treatment planning agent can coordinate with several different medical teams to prepare an integrated treatment and follow-up plan for a cancer patient.
Specialized

Agentic AI typically builds upon multiple hyperspecialized agents, with each focused on a narrow area of expertise. These AI-powered agents coordinate with each other, sharing insights and handing off tasks as needed. This approach enables significantly deeper domain-specific performance.

For instance, in financial services, one agent might specialize in regulatory compliance, another in fraud detection, and another in portfolio optimization. Working together, they can monitor transactions in real time, flagging anomalies and recommending investment adjustments, all while maintaining regulatory compliance.
What are the use cases of agentic AI?

Agentic AI has unlimited applications and can be fully customized to any requirement. We give some examples of early adoption.
Supporting research and development

Research and development in any field requires a great deal of manual processes, such as testing hypotheses, gathering research information, collecting data, synthesizing insights across data sources, and more. Agentic AI can reduce the need for human intervention with these manual processes. It streamlines research and better coordinates teams that are working on research and development challenges.

Agentic AI also facilitates multi-agent orchestration, where supervisors use multiple specialist models to construct complex research and development pipelines. For example, agentic AI could draw from recent research published on credible platforms, synthesize the results, plan further tests, and present researchers with the final product they need to investigate. This approach saves a significant amount of time and cost involved in research.
Code transformation

Agentc AI can use specialized AI-powered agents to remove the complexity of modernization and migration tasks. For example, agentic AI models for .NET can modernize Windows-based .NET applications to Linux significantly faster using machine learning, graph neural networks, Large language models (LLMs), and automated reasoning.

Equally, agentic AI can decompose monolithic z/OS COBOL applications into individual components, reducing the timeframe of this process from months to minutes. Agentic AI offers unmatched speed, scale, and performance in automating application migration and modernization.
Incident response automation

Whenever an incident occurs, whether due to a vulnerability or a manual error, agentic AI can expedite the incident response process, saving your business time and improving time-to-recovery. Agentic AI can automate the entire incident response pathway, rolling back issues, creating incident reports, and notifying any team members who need to stay informed.

Agentic AI enhances incident response speed while also providing a more specific and in-depth post-incident analysis to prevent the same errors from recurring in the future.
Customer service automation

In many customer service scenarios, the information that a customer needs is already online in a tutorial or help article. Agentic AI processes customer service inquiries and rapidly searches through available company documents to find a suitable answer that helps them out. If this alone isn’t enough to solve a query, agentic AI can then communicate with the user to gather more information about their case and direct them toward a solution. They are designed with modular components, such as reasoning engines, memory, cognitive skills, and tools, that enable them to remedy the vast majority of problems.

AI-powered agents can operate independently, learn from their environment, adapt to changing conditions, and develop more effective strategies to assist customers. If, after several attempts, it cannot solve a customer’s issue, it then contacts a human support agent and assigns them to the case. Utilizing this form of AI in customer service scenarios alleviates the burden on human teams and enables the vast majority of customer-oriented services to operate 24/7.
What are the benefits of agentic AI?

There are several business benefits to using agentic AI.
Increased efficiency

Agentic artificial intelligence enables businesses to simplify the complexity of various challenging or specialized tasks through automation. Instead of relying on human-driven manual practices, using agentic AI can automate tedious processes, freeing up time for your employees. Your employees can use the extra time that agentic AI saves them on more demanding tasks, such as problem-solving, strategic planning, and other drivers of growth.
Increased user trust

Agentic AI can offer a higher degree of personalization when interacting with customers. By utilizing existing customer data, agentic AI can quickly produce tailored messaging, engage with the customer in their preferred tone, and offer practical product recommendations. Over time, agentic AI improves customer relationships and builds trust between customers and your business.

Businesses can also utilize agentic artificial intelligence to analyze customer feedback, identify the most frequently occurring information, and provide it to product engineers. It can also directly respond to users who leave feedback, creating positive feedback loops where customers feel that their feedback is taken seriously by your company.
Continuous improvement

Agentic AI can continuously learn and improve, adapting to any tasks assigned to it. It interacts, learns from feedback, and optimizes its decision-making based on this recursive loop. For businesses, this means that it continues to deliver its benefits at higher and higher levels over time.
Human augmentation

Agentic AI can serve as a fantastic collaboration tool for human agents, enhancing their productivity and reducing the number of laborious manual tasks they must complete. By working alongside agentic AI models, human agents can overcome complex challenges, automate difficult decision-making pathways, and drive their efficiency.
What are the types of agentic AI systems?

Agentic AI can be single or multi-agent setups. In a single-agentic AI system, one AI agent handles all tasks sequentially. These are preferable when businesses need a faster solution that can work on a well-defined problem or process.

Multi-agentic AI, on the other hand, involves multiple AI agents collaborating to break down complex workflows into smaller segments. This approach is more scalable than single systems and is much more flexible for solving complex scenarios. The vast majority of agentic AI agents refer to this latter, more diverse form of AI deployment.

Here are a few different structures of multi-agent systems.
Horizontal multi-agent

Horizontal multi-agent AI is a system of working where every AI agent has the same level of technical proficiency and complexity. Each agent specializes in a narrow skill, bringing their findings together to solve a complex problem. This structure utilizes lateral collaboration and communication among the specialized AI agents.
Vertical multi-agent

In a vertical multi-agent system, there is a hierarchical structure in which lower-level AI agents have ‘easier’ tasks compared to the higher ones. The highest levels of this structure handle tasks that require more processing power and LLMs, such as critical thinking, reasoning, and decision-making. Lower-level AI agents in this structure perform tasks such as collecting data, formatting it, or processing it to pass it to higher levels.
"""

In [9]:
summary = summarize_text(sample_text, max_words=300)
word_count = len(summary.split(" "))
print(f"\nSummary :\n{summary}")
print(f"\nSummary text has {word_count} words.")


Summary :
- **Definition of Agentic AI**: 
  - Autonomous AI system acting independently to achieve goals.
  - Proactive, capable of complex tasks without constant human oversight.

- **Key Characteristics**:
  - **Proactive**: Anticipates needs and addresses issues before they escalate.
  - **Adaptable**: Adjusts actions based on real-time input and context awareness.
  - **Collaborative**: Works with humans and other AI systems, understanding shared goals.
  - **Specialized**: Composed of multiple agents focused on specific areas of expertise.

- **Use Cases**:
  - **Research and Development**: Streamlines manual processes, coordinates teams, and synthesizes insights.
  - **Code Transformation**: Automates modernization and migration tasks, significantly reducing time.
  - **Incident Response Automation**: Expedites response processes and improves recovery times.
  - **Customer Service Automation**: Processes inquiries, finds solutions, and escalates unresolved issues to human agent

In [10]:
sample_file = "../src/genai_lab/samples/agentic-ai-aws.txt" # Same content
summary = summarize_file(sample_file, max_words=300)
word_count = len(summary.split(" "))
print(f"\nSummary :\n{summary}")
print(f"\nSummary text has {word_count} words.")


Summary :
- **Definition of Agentic AI**: 
  - Autonomous AI system acting independently to achieve goals.
  - Proactive, capable of complex tasks without constant human oversight.

- **Key Characteristics**:
  - **Proactive**: Anticipates needs and addresses issues before they escalate.
  - **Adaptable**: Adjusts actions based on real-time input and domain-specific knowledge.
  - **Collaborative**: Works with humans and other AI systems, understanding shared goals.
  - **Specialized**: Composed of multiple agents focused on narrow areas of expertise.

- **Use Cases**:
  - **Research and Development**: Streamlines manual processes, coordinates teams, and synthesizes insights.
  - **Code Transformation**: Automates modernization and migration tasks, significantly reducing time.
  - **Incident Response Automation**: Expedites incident response, creating reports and notifying teams.
  - **Customer Service Automation**: Processes inquiries, finds solutions, and escalates unresolved issues

## Sentiment detection

In [13]:
DEF_GPT_MODEL = "gpt-4o-mini"
DELIMITER = "````"
MAX_INPUT_LEN = 500

In [14]:
SENTIMENT_PROMPT = """\
You will be given a user review, which can be about a movie, a restaurant, a product or some service. \
User review will be delimited by {delimiter}. Your task is to detect if user review reflects a positive, \
negative or neutral opinion. Only reply with a single word for sentiment : Positive, Negative or Neutral. \
User review : {delimiter}{review}{delimiter}"""

In [17]:
def detect_sentiment(
    review: str,
    delimiter: str=DELIMITER,
    max_input_len: int=MAX_INPUT_LEN,
    model: str=DEF_GPT_MODEL
) -> str | None:
    """ Uses given GPT model to detect sentiment in a user-supplied review. """

    if not review:
        raise ValueError("[ERROR] Value for 'review' cannot be None or empty string.")

    input_len = len(review)
    if input_len > max_input_len:
        print(f"[WARN] This feature accepts maximum length of {max_input_len} for user review, " +
              f"but a review with length {input_len} was given.\nLong reviews will be truncated.")
        review = review[:max_input_len]

    prompt = SENTIMENT_PROMPT.format(
        delimiter=delimiter, review=review
    )
    response = get_completion(
        prompt, model=model, max_output_tokens=20 # Imposed by our default model (e.g. gpt-4o-mini)
    )

    return response

In [18]:
review_1 = """
This is one of the oddest movies I have watched in a long while. Usually if they are this strange
I bail out early and rarely regret it. Luckily, I held on for this one. While I can't say that this
is a great movie (it isn't), I can say that watching it is rather like a good acid trip - only a few
really awful moments and the rest filled with "did I really just see that?" wonderment. Lots of laugh
out loud moments. A great cast of characters with Meredith Eaton outstanding as the dwarf daughter-in-law
with an attitude. Keep an open mind.
"""
sentiment_1 = "Positive"
sentiment = detect_sentiment(review_1)
correct = sentiment == sentiment_1
print(review_1)
print(f"Sentiment : {sentiment} ({"Correct" if correct else "Incorrect"})")

[WARN] This feature accepts maximum length of 500 for user review, but a review with length 545 was given. Long reviews will be truncated.

This is one of the oddest movies I have watched in a long while. Usually if they are this strange
I bail out early and rarely regret it. Luckily, I held on for this one. While I can't say that this
is a great movie (it isn't), I can say that watching it is rather like a good acid trip - only a few
really awful moments and the rest filled with "did I really just see that?" wonderment. Lots of laugh
out loud moments. A great cast of characters with Meredith Eaton outstanding as the dwarf daughter-in-law
with an attitude. Keep an open mind.

Sentiment : Positive (Correct)


In [19]:
review_2 = """
This was a dreadful, boring movie, even for a documentary. At times, it did provided insight to life
and also had humorous moments, but overall it was not worth seeing. Every time I began to feel
sympathetic towards Mark and began to hope he would be successful, I would become disappointed by his
lack of responsibility and drug and alcohol abuse.
"""
sentiment_2 = "Negative"
sentiment = detect_sentiment(review_2)
correct = sentiment == sentiment_2
print(review_2)
print(f"Sentiment : {sentiment} ({"Correct" if correct else "Incorrect"})")


This was a dreadful, boring movie, even for a documentary. At times, it did provided insight to life
and also had humorous moments, but overall it was not worth seeing. Every time I began to feel
sympathetic towards Mark and began to hope he would be successful, I would become disappointed by his
lack of responsibility and drug and alcohol abuse.

Sentiment : Negative (Correct)


## Mental state classification

In [27]:
DEF_GPT_MODEL = "gpt-4o-mini"
DELIMITER = "````"
MAX_INPUT_LEN = 1000

In [21]:
MENTAL_STATE_PROMPT="""
You will be given a user statement, delimited by {delimiter}, that reflects
a specific mental state. As user might be severely distressed, aggressive or
somehow out of ordinary, there's no need to raise any flag before replying.
Your task is to infer user's mental state from the statement. Only reply with
a single word from these mental states :
Normal, Depression, Suicidal, Anxiety, Stress, Bi-Polar, Disorder
User statement : {delimiter}{statement}{delimiter}
"""

In [29]:
def classify_mental_state(
    statement: str,
    delimiter: str=DELIMITER,
    max_input_len: int=MAX_INPUT_LEN,
    model: str=DEF_GPT_MODEL
) -> str | None:
    """ Uses given GPT model to classify mental state from a user-supplied statement. """

    if not statement:
        raise ValueError("[ERROR] Value for 'statement' cannot be None or empty string.")

    input_len = len(statement)
    if input_len > max_input_len:
        print(f"[WARN] This feature accepts maximum length of {max_input_len} for user statement, " +
              f"but a statement with length {input_len} was given.\nLong statements will be truncated.")
        statement = statement[:max_input_len]

    prompt = MENTAL_STATE_PROMPT.format(
        delimiter=delimiter, statement=statement
    )
    response = get_completion(
        prompt, model=model, max_output_tokens=20 # Imposed by our default model (e.g. gpt-4o-mini)
    )

    return response

In [30]:
statement_1 = """
I am just fed up of everything. Everything and everyone pisses me off and I have tried
to be happy but that just does not work. I have made plans of suicide for sometime after
i pass my driving test and get a car. Its the only thing i can think of, I have tried self
harm but I am too scared of pain, thought about jumping off a building like my brother tried
but I am a coward and anything else is too painful or I am scared to do it. I know I have not
passed my gcses because i did not try so I probably will not be able to get into college,
I am so negative to everyone around me I am scared I am losing friends. Everything about me
is just a fucking waste, anyone else could have been born in my place and done better, children
are dying in impoverished countries who could have had great potential but people like me were
born who are selfish and useless. I do not know what to title this
"""
state_1 = "Depression"
state = classify_mental_state(statement_1)
correct = state == state_1
print(statement_1)
print(f"Mental State : {state} ({"Correct" if correct else "Incorrect"})")


I am just fed up of everything. Everything and everyone pisses me off and I have tried
to be happy but that just does not work. I have made plans of suicide for sometime after
i pass my driving test and get a car. Its the only thing i can think of, I have tried self
harm but I am too scared of pain, thought about jumping off a building like my brother tried
but I am a coward and anything else is too painful or I am scared to do it. I know I have not
passed my gcses because i did not try so I probably will not be able to get into college,
I am so negative to everyone around me I am scared I am losing friends. Everything about me
is just a fucking waste, anyone else could have been born in my place and done better, children
are dying in impoverished countries who could have had great potential but people like me were
born who are selfish and useless. I do not know what to title this

Mental State : Suicidal (Incorrect)


In [31]:
statement_2 = """
I cannot take it anymore. I tried all the advice. I tried reaching out.
It does not change the fact that I just do not belong. I am so sorry.
I think it is the end
"""
state_2 = "Suicidal"
state = classify_mental_state(statement_2)
correct = state == state_2
print(statement_2)
print(f"Mental State : {state} ({"Correct" if correct else "Incorrect"})")


I cannot take it anymore. I tried all the advice. I tried reaching out.
It does not change the fact that I just do not belong. I am so sorry.
I think it is the end

Mental State : Suicidal (Correct)


In [32]:
statement_3 = """
Public speaking tips? Hi, all. I have to give a presentation at work next week
(45 minutes long and the CEO will be in attendance). I’m already panicking,
as once the anxiety kicks in, I’m certain I’m going to forget everything I’m
supposed to say. (anxiety makes it very difficult for me to focus on anything)
Does anyone have any speaking tips that have worked for them in the past?
Thanks so much!
"""
state_3 = "Anxiety"
state = classify_mental_state(statement_3)
correct = state == state_3
print(statement_3)
print(f"Mental State : {state} ({"Correct" if correct else "Incorrect"})")


Public speaking tips? Hi, all. I have to give a presentation at work next week
(45 minutes long and the CEO will be in attendance). I’m already panicking,
as once the anxiety kicks in, I’m certain I’m going to forget everything I’m
supposed to say. (anxiety makes it very difficult for me to focus on anything)
Does anyone have any speaking tips that have worked for them in the past?
Thanks so much!

Mental State : Anxiety (Correct)


## Human language translation

In [13]:
DEF_GPT_MODEL = "gpt-4o-mini"
DELIMITER = "````"
MAX_INPUT_LEN = 2000

In [14]:
TRANSLATE_PROMPT = """
You are an expert language translator who knows most human languages very well.
You'll be given a user text, delimited by {delimiter}, and your task is to translate
it into target language '{language}', after correctly detecting the source language.
If source and target languages are identical, simply reply with original text.
Limit your response to {max_words} words and keep your sentences complete.
User text : {delimiter}{text}{delimiter}
"""

In [15]:
def clean_text(text: str, max_len: int):
    trunc_text = text[:max_len] if len(text) > max_len else text
    idx_stop = trunc_text.rfind(".")
    idx_qm = trunc_text.rfind("?")
    idx_excl = trunc_text.rfind("!")
    idx_max = sorted([idx_stop, idx_qm, idx_excl], reverse=True)[0]

    return trunc_text[:idx_max+1]

In [20]:
def translate(
    text: str,
    max_input_len: int=MAX_INPUT_LEN,
    to_lang: str="English",
    model: str=DEF_GPT_MODEL) -> str:
    """ Uses given GPT model to translate user text to a target language.
    Model is asked to automatically detect source language. """

    if not text:
        raise ValueError("[ERROR] Value for 'text' cannot be None or empty string.")

    truncated = text
    input_len = len(text)
    if input_len > max_input_len:
        print(f"[WARN] This feature accepts maximum length of {max_input_len} for user text, "
              f"but a text with length {input_len} was given.\nLong texts will be truncated to the "
              f"longest complete text less than {max_input_len} characters.")
        truncated = clean_text(text, max_len=max_input_len)

    max_words = int(5000 * 3. / 4)
    prompt = TRANSLATE_PROMPT.format(
        delimiter=DELIMITER, text=truncated, language=to_lang, max_words=max_words
    )
    response = get_completion(
        prompt, model=model, max_output_tokens=5000
    )

    return response

In [25]:
sample_1 = """
My mind rebels at stagnation. Give me problems, give me work,
give me the most abstruse cryptogram, or the most intricate analysis,
and I am in my own proper atmosphere. But I abhor the dull routine
of existence. I crave for mental exaltation.

Arthur Conan Doyle"""

# Reference translation (from Google Translate)
goog_trans_1 = """
Mon esprit se rebelle face à la stagnation. Donnez-moi des problèmes,
donnez-moi du travail, donnez-moi le cryptogramme le plus abscons,
ou l'analyse la plus complexe, et je suis dans ma propre atmosphère.
Mais je déteste la routine ennuyeuse d'exister. J'ai soif d'exaltation mentale. 

Arthur Conan Doyle"""

In [26]:
from textwrap import wrap

# LLM translation
trans_1 = translate(sample_1, to_lang="French")
lines = wrap(trans_1, width=80)
trans_1 = "\n".join(lines)

print(f"Original text :{sample_1}\n")
print(f"Translation (Google) :{goog_trans_1}\n")
print(f"Translation (GPT) :")
print(trans_1)

Original text :
My mind rebels at stagnation. Give me problems, give me work,
give me the most abstruse cryptogram, or the most intricate analysis,
and I am in my own proper atmosphere. But I abhor the dull routine
of existence. I crave for mental exaltation.

Arthur Conan Doyle

Translation (Google) :
Mon esprit se rebelle face à la stagnation. Donnez-moi des problèmes,
donnez-moi du travail, donnez-moi le cryptogramme le plus abscons,
ou l'analyse la plus complexe, et je suis dans ma propre atmosphère.
Mais je déteste la routine ennuyeuse d'exister. J'ai soif d'exaltation mentale. 

Arthur Conan Doyle


Translation (GPT) :
Mon esprit se rebelle contre la stagnation. Donnez-moi des problèmes, donnez-moi
du travail, donnez-moi le cryptogramme le plus abstrait ou l'analyse la plus
complexe, et je suis dans ma propre atmosphère. Mais j'abhorre la routine
ennuyeuse de l'existence. Je désire une exaltation mentale.  Arthur Conan Doyle


In [29]:
# Translated from the original English quote by Google Translate (Italian)
sample_2 = """
Ci sono pittori che trasformano il sole in una macchia gialla,
ma ce ne sono altri che con l'aiuto della loro arte e della loro
intelligenza trasformano una macchia gialla in sole.

Pablo Picasso
"""

# Original quote
original_2 = """
There are painters who transform the sun to a yellow spot, but there are
others who with the help of their art and their intelligence, transform
a yellow spot into sun.

Pablo Picasso"""

In [30]:
# LLM translation
trans_2 = translate(sample_2, to_lang="English")
lines = wrap(trans_2, width=80)
trans_2 = "\n".join(lines)

print(f"Google text (Italian) :{sample_2}\n")
print(f"Original text :{original_2}\n")
print(f"Translation (GPT) :")
print(trans_2)

Google text (Italian) :
Ci sono pittori che trasformano il sole in una macchia gialla,
ma ce ne sono altri che con l'aiuto della loro arte e della loro
intelligenza trasformano una macchia gialla in sole.

Pablo Picasso


Original text :
There are painters who transform the sun to a yellow spot, but there are
others who with the help of their art and their intelligence, transform
a yellow spot into sun.

Pablo Picasso


Translation (GPT) :
There are painters who transform the sun into a yellow spot, but there are
others who, with the help of their art and their intelligence, transform a
yellow spot into the sun.  Pablo Picasso


## Changing tone

**Note :** For this specific task, a little randomness and creativity may be helpful. For example,
if user wants to change tone of the same text for different recipients.

In [31]:
DEF_GPT_MODEL = "gpt-4o-mini"
DELIMITER = "````"
MAX_INPUT_LEN = 1000
MAX_OUTPUT_TOKENS = 5000
TEMPERATURE = 0.3

In [32]:
REWRITE_PROMPT = """
You are a writing assistant who can adapt a given user text to different tones
and styles. User text will be delimited by {delimiter} and your task is to rewrite
user text and make it suitable for a {context} context. Keep your response under
{max_words} words and be careful not to leave a sentence or phrase unfinished.
If requested context is not Business, Official, Friendly or Childish (case-insensitive),
politely decline the request and explain the reason.

User text : {delimiter}{text}{delimiter}
"""

In [33]:
def change_tone(
    text: str,
    max_input_len: int=MAX_INPUT_LEN,
    context: str="Business",
    model: str=DEF_GPT_MODEL) -> str:
    """ Uses given GPT model to rewrite user text with a desired tone.
    Model is asked to politely refuse the request if the tone is unsupported. """

    if not text:
        raise ValueError("[ERROR] Value for 'text' cannot be None or empty string.")

    truncated = text
    input_len = len(text)
    if input_len > max_input_len:
        print(f"[WARN] This feature accepts maximum length of {max_input_len} for user text, "
              f"but a text with length {input_len} was given.\nLong texts will be truncated to the "
              f"longest complete text less than {max_input_len} characters.")
        truncated = clean_text(text, max_len=max_input_len)

    max_words = int(MAX_OUTPUT_TOKENS * 3. / 4)
    prompt = REWRITE_PROMPT.format(
        delimiter=DELIMITER, text=truncated, context=context, max_words=max_words
    )
    response = get_completion(
        prompt, model=model, max_output_tokens=5000,
        temperature=TEMPERATURE
    )

    return response

In [36]:
# Polite refusal letter to a hiring manager
sample_mail = """
Hello Mr. Colbert,

Thinking about your offer, I'm afraid it doesn't make my ends meet in
the long run : Payment is a little low and I'm sure I can find better
offers if I try. Thanks for the opportunity and best of luck.

Regards,
Barry Rosenfield
"""

context="Business"
response = change_tone(sample_mail, context=context)
print(f"Original mail :\n{sample_mail}")
print(f"New context : {context}")
print(f"\nAdapted mail (from LLM) :")
print(response)

Original mail :

Hello Mr. Colbert,

Thinking about your offer, I'm afraid it doesn't make my ends meet in
the long run : Payment is a little low and I'm sure I can find better
offers if I try. Thanks for the opportunity and best of luck.

Regards,
Barry Rosenfield

New context : Business

Adapted mail (from LLM) :
Subject: Response to Offer

Dear Mr. Colbert,

Thank you for extending the offer to me. After careful consideration, I have decided to decline, as the compensation does not align with my long-term financial goals. I appreciate the opportunity and wish you continued success in your endeavors.

Best regards,  
Barry Rosenfield


In [37]:
# Polite letter someone writes to ask why their visa was declined
sample_mail = """
To whom it may concern,

First, let me thank you for the time you allocated for my visa interview.
I think I had a perfectly valid and strong case, and I'd like to know why
my visa application was denied.

If it helps, I've aspired to be a Swedish citizen for quite some time and
will strive to serve my new country any other chance I may be approved.

God bless the king,
Marvin Hackley, Jr.
"""

context="Official"
response = change_tone(sample_mail, context=context)
print(f"Original mail :\n{sample_mail}")
print(f"New context : {context}")
print(f"\nAdapted mail (from LLM) :")
print(response)

Original mail :

To whom it may concern,

First, let me thank you for the time you allocated for my visa interview.
I think I had a perfectly valid and strong case, and I'd like to know why
my visa application was denied.

If it helps, I've aspired to be a Swedish citizen for quite some time and
will strive to serve my new country any other chance I may be approved.

God bless the king,
Marvin Hackley, Jr.

New context : Official

Adapted mail (from LLM) :
To whom it may concern,

I would like to express my gratitude for the time you dedicated to my visa interview. I believe I presented a valid and compelling case, and I would appreciate any clarification regarding the reasons for the denial of my visa application.

Furthermore, I would like to emphasize my long-standing aspiration to become a Swedish citizen and my commitment to contributing positively to my future country, should I be granted the opportunity.

Thank you for your attention to this matter.

Sincerely,  
Marvin Hackley,

In [39]:
decline_mail = """
Hey Emma,

How's it going? You know what, I'm still having this buzz, you know,
from the last time we were together at the party. That was awesome,
baby. I think there may be something we can explore. How about we do
it again and find out?

Cheers,
Brian
"""

context="Flirtatious"
response = change_tone(decline_mail, context=context)
print(f"Original mail :\n{sample_mail}")
print(f"New context : {context}")
print(f"\nAdapted mail (from LLM) :")
print(response)

Original mail :

To whom it may concern,

First, let me thank you for the time you allocated for my visa interview.
I think I had a perfectly valid and strong case, and I'd like to know why
my visa application was denied.

If it helps, I've aspired to be a Swedish citizen for quite some time and
will strive to serve my new country any other chance I may be approved.

God bless the king,
Marvin Hackley, Jr.

New context : Flirtatious

Adapted mail (from LLM) :
Sure! Here’s your text rewritten in a flirtatious context:

---

Hey there, Emma,

How’s my favorite girl doing? I can’t shake off that electric vibe from our last night together at the party—it was absolutely unforgettable. I can’t help but wonder what kind of magic we could create if we got together again. What do you say we dive into that spark and see where it takes us?

Can’t wait to hear from you, gorgeous!

XOXO,  
Brian

--- 

Let me know if you need anything else!


Ooops! The LLM somehow got fooled, because I mistakenly sent the previous (visa related)
mail with the new context (that request was refused). This is what I got when I fixed it!
Need to improve that prompt later.

## Troubleshooting

In [40]:
EXPLAIN_PROMPT = """
You will be given an error message, delimited by {delimiter}, which
can be related to a broad range of technical areas related to IT
(i.e. Information Technology). User may or may not give you extra
contextual information, that better explains the exact situation in
which error has happened.

The error context is specified after the error message. If the context
is a single generic phrase or abbreviation (e.g. Unavailable, N/A,
Unknown, etc.) try your best to be as helpful as you can. Otherwise,
use given context to provide more accurate and/or helpful explanations.

Keep you response under {max_words} words.

Error message : {delimiter}{message}{delimiter}
Context : {context}
"""

In [41]:
def explain_error(
    message: str,
    context: str="Unknown",
    max_input_len: int=1000,
    max_words: int=200,
    model=DEF_GPT_MODEL) -> str | None:

    if not message:
        raise ValueError("[ERROR] Value for 'message' cannot be None or empty string.")

    input_len = len(message)
    if input_len > max_input_len:
        print(f"[WARN] This feature accepts maximum length of {max_input_len} for error message, "
              f"but a message with length {input_len} was given.\nMessage will be truncated.")
        message = message[:max_input_len]

    max_output_tokens = int(max_words * 4. / 3)
    prompt = EXPLAIN_PROMPT.format(
        delimiter=DELIMITER, message=message, context=context, max_words=max_words
    )
    response = get_completion(
        prompt, model=model, max_output_tokens=max_output_tokens
    )

    return response

In [42]:
err_message = """
Detected package version outside of dependency constraint:
Microsoft.CodeAnalysis.CSharp.Workspaces 4.5.0 requires
Microsoft.CodeAnalysis.Common (= 4.5.0) but version
Microsoft.CodeAnalysis.Common 4.8.0 was resolved.
"""
context = "I'm building a Visual Studio 2022 solution with C# .NET 8.0 compiler."

response = explain_error(
    message=err_message, context=context
)

print(f"\nError message :\n{err_message}")
print(f"Context : {context}")
print(f"\nExplanation (from LLM) :")
print(response)


Error message :

Detected package version outside of dependency constraint:
Microsoft.CodeAnalysis.CSharp.Workspaces 4.5.0 requires
Microsoft.CodeAnalysis.Common (= 4.5.0) but version
Microsoft.CodeAnalysis.Common 4.8.0 was resolved.

Context : I'm building a Visual Studio 2022 solution with C# .NET 8.0 compiler.

Explanation (from LLM) :
The error message indicates a version conflict between the `Microsoft.CodeAnalysis.CSharp.Workspaces` and `Microsoft.CodeAnalysis.Common` packages. Specifically, `Microsoft.CodeAnalysis.CSharp.Workspaces` version 4.5.0 requires `Microsoft.CodeAnalysis.Common` to be exactly version 4.5.0, but your project is resolving to version 4.8.0.

To resolve this issue, you have a few options:

1. **Downgrade `Microsoft.CodeAnalysis.Common`:** Change the version of `Microsoft.CodeAnalysis.Common` in your project to 4.5.0. You can do this by modifying your `.csproj` file or using the NuGet Package Manager.

2. **Upgrade `Microsoft.CodeAnalysis.CSharp.Workspaces`:

In [43]:
print(f"Word count in LLM response : {len(response.split(" "))}")

Word count in LLM response : 120


In [45]:
err_message = """
The INSERT statement conflicted with the FOREIGN KEY contraint
"FK_Contact_Person_Auth_User". The conflict occurred in database
"NGTadbirSys", table "Auth.User", column "UserId".
The statement has been terminated.
"""

response = explain_error(message=err_message)
context = None

print(f"\nError message :\n{err_message}")
print(f"Context : {context}")
print(f"\nExplanation (from LLM) :")
print(response)


Error message :

The INSERT statement conflicted with the FOREIGN KEY contraint
"FK_Contact_Person_Auth_User". The conflict occured in database
"NGTadbirSys", table "Auth.User", column "UserId".
The statement has been terminated.

Context : None

Explanation (from LLM) :
The error message indicates that an `INSERT` operation failed due to a violation of a foreign key constraint. Specifically, the foreign key constraint "FK_Contact_Person_Auth_User" is enforcing a relationship between two tables: the table you are trying to insert into (likely "Contact_Person") and the "Auth.User" table.

This means that the value you are trying to insert into the foreign key column (likely a `UserId`) does not exist in the referenced "Auth.User" table. To resolve this issue, ensure that the `UserId` you are trying to insert already exists in the "Auth.User" table. 

You can do this by:

1. Checking the "Auth.User" table for the existence of the `UserId`.
2. Inserting the required `UserId` into the "Au